# 🎙️ Fine-Tuning de Whisper para Reconocimiento de Voz Multilingüe

**Objetivo:** fine-tuning de **Whisper** (`openai/whisper-tiny` o `small`) para
transcripción automática (ASR) de lenguas de bajo recurso: **español, italiano
y suajili** (extensible a wolof, fula, bambara...).

**Diseño del pipeline:**
1. Preprocesado de audio (MP3 → WAV 16 kHz, duración fija)
2. Preparación de transcripciones (texto limpio + token de lengua)
3. Dataset PyTorch + DataLoader
4. Fine-tuning de Whisper (100% PyTorch, AMP)
5. Evaluación con **WER / CER** (jiwer)
6. Inferencia sobre un archivo de audio

**Requisitos:** Python 3.10+, PyTorch, `transformers`, `torchaudio`,
`librosa`, `soundfile`, `jiwer`, `datasets`.

> ⚠️ **Modelos gated:** si el modelo requiere autenticación, define
> `export HF_TOKEN=hf_...` en el entorno. Nunca se hardcodean tokens.

---
## 0. Instalación

```bash
pip install torch transformers torchaudio librosa soundfile jiwer tqdm pandas
sudo apt-get install -y ffmpeg   # para decodificar MP3/OGG
```

Si tu audio ya está en WAV 16 kHz, el paso 2 es opcional.

In [ ]:
# === IMPORTS Y DISPOSITIVO ===
import os
import json
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torchaudio
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"torchaudio: {torchaudio.__version__}")
print(f"HF_TOKEN definido: {bool(os.environ.get('HF_TOKEN'))}")

---
## 1. Configuración

`DATA_DIR` apunta a la carpeta de datos con esta estructura:

```
DATA_DIR/
├── audio/
│   ├── es/  *.wav
│   ├── it/  *.wav
│   └── sw/  *.wav
└── transcriptions/
    ├── es.json     [{"audio_file": "...", "text": "..."}]
    ├── it.json
    └── sw.json
```

Alternativamente, se pueden cargar ficheros `.pt` ya preprocesados (ver paso 4).

In [ ]:
# === CONFIGURACIÓN ===
DATA_DIR = Path("./data/sst")          # <- CAMBIA ESTO
OUTPUT_DIR = Path("./models/whisper_multilingue")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class Config:
    # --- Modelo ---
    model_name: str = "openai/whisper-tiny"   # tiny (rápido) | small (mejor) | base
    language: str = "multilingual"            # o "wolof" para una sola lengua

    # --- Lenguas ---
    # token de lengua <|xx|> que se añade al tokenizador y se antepone a las etiquetas
    languages: Dict[str, str] = field(default_factory=lambda: {
        "spanish": "<|es|>",
        "italian": "<|it|>",
        "swahili": "<|sw|>",
    })

    # --- Audio ---
    sample_rate: int = 16000
    target_length_s: float = 5.0              # padding/truncado del audio

    # --- Entrenamiento ---
    num_epochs: int = 4
    batch_size: int = 16
    learning_rate: float = 1e-4
    gradient_accumulation_steps: int = 4
    warmup_steps: int = 50
    max_label_length: int = 256
    val_split: float = 0.1
    seed: int = 42

    # --- Generación ---
    num_beams: int = 5


config = Config()
print(config)

---
## 2. (Opcional) Preprocesado de audio

Convierte cualquier formato a **WAV mono 16 kHz** y recorta/rellena a
`target_length_s` segundos. Whisper trabaja a 16 kHz; normalizar la duración
simplifica el batching.

In [ ]:
# === PREPROCESADO DE AUDIO (OPCIONAL) ===
import librosa
import soundfile as sf


def preprocess_audio_file(audio_path: Path, out_dir: Path,
                          sample_rate=16000, target_length_s=5.0) -> Optional[Path]:
    """Convierte a WAV mono 16 kHz con duración fija. Devuelve la ruta o None."""
    try:
        waveform, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
        target_samples = int(sample_rate * target_length_s)
        if len(waveform) < target_samples:
            waveform = np.pad(waveform, (0, target_samples - len(waveform)))
        else:
            waveform = waveform[:target_samples]
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / (audio_path.stem + ".wav")
        if not out_path.exists():
            sf.write(out_path, waveform, sample_rate)
        return out_path
    except Exception as e:
        print(f"⚠️ Error con {audio_path}: {e}")
        return None


def preprocess_language(lang: str, audio_dir: Path, wav_dir: Path):
    """Preprocesa todos los audios de una lengua."""
    valid_ext = (".mp3", ".wav", ".flac", ".ogg", ".m4a", ".aac")
    files = [f for f in Path(audio_dir).glob("*") if f.suffix.lower() in valid_ext]
    ok = 0
    for f in tqdm(files, desc=f"Preprocesando {lang}"):
        if preprocess_audio_file(f, wav_dir, config.sample_rate, config.target_length_s):
            ok += 1
    print(f"✅ {lang}: {ok}/{len(files)} audios -> {wav_dir}")


# Descomenta para ejecutar el preprocesado:
# preprocess_language("es", DATA_DIR / "audio/raw/es", DATA_DIR / "audio/es")
# preprocess_language("it", DATA_DIR / "audio/raw/it", DATA_DIR / "audio/it")
# preprocess_language("sw", DATA_DIR / "audio/raw/sw", DATA_DIR / "audio/sw")

---
## 3. (Opcional) Preparación de transcripciones

Cada fichero JSON de transcripciones tiene la forma:

```json
[{"audio_file": "ejemplo_001.wav", "text": "buenos días, ¿cómo estás?"}]
```

`prepare_transcriptions()` limpia el texto (minúsculas, espacios) y construye
una lista unificada con el token de lengua de cada muestra.

In [ ]:
# === PREPARACIÓN DE TRANSCRIPCIONES (OPCIONAL) ===
import re


def clean_transcript(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def prepare_transcriptions(lang: str, json_path: Path, audio_dir: Path) -> List[dict]:
    """Carga JSON de transcripciones, limpia y verifica que el audio existe."""
    with open(json_path, encoding="utf-8") as f:
        rows = json.load(f)
    items = []
    for row in rows:
        audio_file = Path(audio_dir) / row["audio_file"]
        if not audio_file.exists():
            continue
        text = clean_transcript(row.get("text", ""))
        if not text:
            continue
        items.append({"audio_file": str(audio_file), "text": text, "lang": lang})
    print(f"✅ {lang}: {len(items)} muestras válidas")
    return items


# Descomenta si no tienes los .pt ya preprocesados:
# all_items = []
# for lang, token in config.languages.items():
#     all_items += prepare_transcriptions(
#         lang, DATA_DIR / "transcriptions" / f"{lang}.json", DATA_DIR / "audio" / lang)
# print(f"Total muestras: {len(all_items)}")

---
## 4. Carga de datos preprocesados (`.pt`)

Si ya existen ficheros `.pt` por lengua (lista de dicts con `audio_file` y
`labels` tokenizadas), se cargan directamente y se filtran las muestras cuyo
audio ya no existe. Esta es la ruta recomendada para reutilizar preprocesados.

In [ ]:
# === CARGA DE .pt PREPROCESADOS ===
def load_preprocessed(lang: str, pt_path: Path, audio_dir: Path) -> List[dict]:
    """Carga un .pt por lengua y filtra audios desaparecidos."""
    data = torch.load(pt_path, map_location="cpu")
    kept = []
    for item in data:
        audio_file = Path(audio_dir) / item["audio_file"]
        if audio_file.exists():
            item = dict(item)
            item["audio_file"] = str(audio_file)
            item["lang"] = lang
            kept.append(item)
    print(f"✅ {lang}: {len(kept)}/{len(data)} muestras")
    return kept


# --- Procesador Whisper (tokenizador + feature extractor) ---
processor = WhisperProcessor.from_pretrained(config.model_name)

# Añadir tokens de lengua al tokenizador
lang_tokens = list(config.languages.values())
processor.tokenizer.add_tokens(lang_tokens)
print(f"Tokens de lengua añadidos: {lang_tokens}")
print(f"Tamaño del vocab: {len(processor.tokenizer)}")


# --- Cargar datos por lengua (ajusta rutas a tu estructura) ---
preprocessed = []
for lang, token in config.languages.items():
    preprocessed += load_preprocessed(
        lang,
        DATA_DIR / "preprocessed" / f"preprocessed_{lang}.pt",
        DATA_DIR / "audio" / lang,
    )
print(f"Total muestras: {len(preprocessed)}")

---
## 5. Dataset y DataLoader

- `__getitem__`: carga el WAV (resample a 16 kHz si hace falta), extrae
  `input_features` con el procesador y antepone el **token de lengua** a las
  etiquetas: `<|es|> buenos días ...`
- `collate_fn`: padding de `input_features` (0.0) y de labels
  (`pad_token_id`); las muestras con error se descartan.

In [ ]:
# === DATASET ===
class AudioTextDataset(Dataset):
    def __init__(self, items: List[dict], processor, languages: Dict[str, str],
                 sample_rate=16000):
        self.items = items
        self.processor = processor
        self.languages = languages
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        try:
            waveform, sr = torchaudio.load(item["audio_file"])
            if sr != self.sample_rate:
                resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
                waveform = resampler(waveform)
            features = self.processor(
                waveform.squeeze(0).numpy(), sampling_rate=self.sample_rate,
                return_tensors="pt").input_features.squeeze(0)

            # Anteponer token de lengua a las etiquetas
            lang_token = self.languages[item["lang"]]
            token_id = self.processor.tokenizer.convert_tokens_to_ids(lang_token)
            labels = torch.cat([torch.tensor([token_id]), item["labels"]])

            return {"input_features": features, "labels": labels}
        except Exception as e:
            print(f"⚠️ Error cargando {item['audio_file']}: {e}")
            return None


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    features = torch.nn.utils.rnn.pad_sequence(
        [b["input_features"] for b in batch], batch_first=True, padding_value=0.0)
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch], batch_first=True,
        padding_value=processor.tokenizer.pad_token_id)
    return {"input_features": features, "labels": labels}


# --- Partición train/val ---
random.seed(config.seed)
random.shuffle(preprocessed)
n_val = int(len(preprocessed) * config.val_split)
val_items, train_items = preprocessed[:n_val], preprocessed[n_val:]

train_ds = AudioTextDataset(train_items, processor, config.languages)
val_ds = AudioTextDataset(val_items, processor, config.languages)
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                          collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                        collate_fn=collate_fn)
print(f"Train: {len(train_items)} | Val: {len(val_items)}")

---
## 6. Modelo

Se carga Whisper y se **redimensionan los embeddings** tras añadir los tokens
de lengua. Se fija `forced_decoder_ids` vacío (los tokens de lengua van en las
etiquetas) y se desactiva el timestamp prediction.

In [ ]:
# === MODELO ===
model = WhisperForConditionalGeneration.from_pretrained(config.model_name)
model.resize_token_embeddings(len(processor.tokenizer))

# Desactivar timestamps y forzar decodificación sin prefijo de tarea
model.config.forced_decoder_ids = None
model.config.suppress_tokens = [token for token in range(0, 50257)
                                if token not in processor.tokenizer.all_special_ids]
model.to(device)
print(f"Parámetros: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

---
## 7. Entrenamiento

Bucle manual en **PyTorch puro**:
- `AdamW` + scheduler lineal con warmup
- **AMP** (`torch.cuda.amp`) en GPU
- Acumulación de gradientes (batch efectivo = `batch_size × grad_accum`)
- Evaluación en validación cada epoch y guardado del mejor checkpoint

In [ ]:
# === ENTRENAMIENTO ===
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
total_steps = (len(train_loader) * config.num_epochs
               // config.gradient_accumulation_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, config.warmup_steps, total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, count = 0.0, 0
    for batch in loader:
        if batch is None:
            continue
        features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)
        labels[labels == processor.tokenizer.pad_token_id] = -100
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                            dtype=torch.float16, enabled=device.type == "cuda"):
            loss = model(input_features=features, labels=labels).loss
        total += loss.item()
        count += 1
    return total / max(count, 1)


def train_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    total, steps = 0.0, 0
    optimizer.zero_grad()
    for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{config.num_epochs}"):
        if batch is None:
            continue
        features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)
        labels[labels == processor.tokenizer.pad_token_id] = -100

        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                            dtype=torch.float16, enabled=device.type == "cuda"):
            loss = model(input_features=features, labels=labels).loss
        loss = loss / config.gradient_accumulation_steps
        scaler.scale(loss).backward()
        total += loss.item() * config.gradient_accumulation_steps
        steps += 1

        if steps % config.gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
    return total / max(steps, 1)


# === BUCLE PRINCIPAL ===
best_val = float("inf")
for epoch in range(config.num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)
    val_loss = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        model.save_pretrained(OUTPUT_DIR / "best")
        processor.save_pretrained(OUTPUT_DIR / "best")
        print(f"💾 Mejor checkpoint: {OUTPUT_DIR / 'best'}")

model.save_pretrained(OUTPUT_DIR / "final")
processor.save_pretrained(OUTPUT_DIR / "final")
print(f"✅ Modelo final en {OUTPUT_DIR / 'final'}")

---
## 8. Evaluación: WER y CER

Se transcribe el conjunto de validación con **beam search** y se calculan
**WER** y **CER** con `jiwer` (word/character error rate; menor = mejor). Los
tokens de lengua se eliminan de las hipótesis antes de comparar.

In [ ]:
# === EVALUACIÓN WER/CER ===
from jiwer import cer, wer


@torch.no_grad()
def transcribe_batch(model, processor, batch):
    features = batch["input_features"].to(device)
    generated = model.generate(
        features,
        max_new_tokens=config.max_label_length,
        num_beams=config.num_beams,
    )
    return processor.tokenizer.batch_decode(generated, skip_special_tokens=True)


def evaluate_wer_cer(model, loader, n_eval=None):
    model.eval()
    hyps, refs = [], []
    for i, batch in enumerate(loader):
        if batch is None:
            continue
        if n_eval and i * config.batch_size >= n_eval:
            break
        texts = transcribe_batch(model, processor, batch)
        for text, labels in zip(texts, batch["labels"]):
            # Referencia: quitar -100 y decodificar
            ref = processor.tokenizer.decode(
                [t for t in labels.tolist() if t != -100], skip_special_tokens=True)
            hyps.append(text.strip().lower())
            refs.append(ref.strip().lower())

    wer_score = wer(refs, hyps) if hyps else float("nan")
    cer_score = cer(refs, hyps) if hyps else float("nan")
    return wer_score, cer_score, hyps, refs


wer_score, cer_score, hyps, refs = evaluate_wer_cer(model, val_loader, n_eval=100)
print("=" * 44)
print(f"WER: {wer_score:.4f}  |  CER: {cer_score:.4f}   (menor = mejor)")
print("=" * 44)
for h, r in list(zip(hyps, refs))[:5]:
    print(f"  REF: {r}")
    print(f"  HYP: {h}")
    print()

---
## 9. Inferencia y demo (Gradio, opcional)

`transcribe_file()` transcribe cualquier WAV con el modelo fine-tuneado. El
bloque Gradio crea una interfaz web para probarlo.

In [ ]:
# === INFERENCIA ===
@torch.no_grad()
def transcribe_file(audio_path: str, model=None, processor=None) -> str:
    """Transcribe un archivo de audio con el modelo fine-tuneado."""
    model = model or WhisperForConditionalGeneration.from_pretrained(
        OUTPUT_DIR / "best").to(device)
    processor = processor or WhisperProcessor.from_pretrained(OUTPUT_DIR / "best")

    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    features = processor(waveform.squeeze(0).numpy(), sampling_rate=16000,
                         return_tensors="pt").input_features.to(device)
    generated = model.generate(features, max_new_tokens=config.max_label_length)
    return processor.tokenizer.decode(generated[0], skip_special_tokens=True)


print(transcribe_file("ejemplo.wav"))  # <- pon aquí tu audio de prueba


# === DEMO GRADIO (opcional) ===
# pip install gradio
# import gradio as gr
# demo = gr.Interface(fn=transcribe_file, inputs=gr.Audio(type="filepath"),
#                     outputs="text", title="ASR multilingüe fine-tuneado")
# demo.launch()

---
## 10. Publicar en Hugging Face Hub (opcional)

Requiere `HF_TOKEN` en el entorno (ver sección 0).

In [ ]:
# === PUSH AL HUB (opcional) ===
def push_to_hub(repo_id: str):
    token = os.environ.get("HF_TOKEN")
    if not token:
        print("⚠️ HF_TOKEN no definido. Ejecuta: export HF_TOKEN=hf_...")
        return
    model.push_to_hub(repo_id, token=token)
    processor.push_to_hub(repo_id, token=token)
    print(f"🚀 Publicado en https://huggingface.co/{repo_id}")


# push_to_hub("tu-usuario/whisper-multilingue-low-resource")  # <- descomenta
print("✅ Notebook completado")